# Statistical Analysis of Heuristic Performance

This notebook computes:
1. Descriptive statistics (mean ± std, min, max) per method over 16 flights
2. Wilcoxon signed-rank tests: each heuristic vs. historical baseline
3. A publication-ready LaTeX table for the paper (Table 6 / Table stats)

**How to use:** Fill in the arrays in Cell 2 with the per-flight
utilisation values from Table 10 of the manuscript (as decimals, e.g. 0.84).

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# ================================================================
# DATA — per-flight volumetric utilisation (0–1 scale)
# Source: Table 11 (historical) and Table 13 (BL/BAF/BLSF)
#         of the manuscript.  Order: flight 1 ... flight 16.
# ================================================================

flight_ids = [
    'F01','F02','F03','F04','F05','F06','F07','F08',
    'F09','F10','F11','F12','F13','F14','F15','F16'
]

# Historical (manual) baselines — from Table 11, "% Volume" column
historical = np.array([
    0.812, 0.811, 0.862, 0.838, 0.833, 0.848, 0.833, 0.825,
    0.854, 0.829, 0.841, 0.852, 0.886, 0.831, 0.834, 0.854
])

# Bottom-Left heuristic — from Table 13, "Bl Model" row
bl = np.array([
    0.886, 0.895, 0.906, 0.889, 0.900, 0.895, 0.901, 0.895,
    0.883, 0.866, 0.883, 0.833, 0.908, 0.889, 0.900, 0.890
])

# Best Area Fit heuristic — from Table 13, "BAF Model" row
baf = np.array([
    0.845, 0.832, 0.808, 0.800, 0.837, 0.831, 0.835, 0.838,
    0.834, 0.836, 0.830, 0.829, 0.803, 0.831, 0.832, 0.840
])

# Best Long Side Fit heuristic — from Table 13, "BLSF Model" row
blsf = np.array([
    0.791, 0.780, 0.820, 0.786, 0.790, 0.800, 0.806, 0.795,
    0.799, 0.828, 0.830, 0.828, 0.690, 0.818, 0.810, 0.834
])

assert len(historical) == 16, 'Need 16 historical values'
assert len(bl) == 16,         'Need 16 BL values'
assert len(baf) == 16,        'Need 16 BAF values'
assert len(blsf) == 16,       'Need 16 BLSF values'

print('Data loaded successfully.')
print(f'Historical mean: {historical.mean():.3f}')
print(f'BL mean:         {bl.mean():.3f}')
print(f'BAF mean:        {baf.mean():.3f}')
print(f'BLSF mean:       {blsf.mean():.3f}')

In [ ]:
# ================================================================
# DESCRIPTIVE STATISTICS
# ================================================================

methods = {
    'Historical (baseline)': historical,
    'Bottom-Left (BL)':      bl,
    'Best Area Fit (BAF)':   baf,
    'Best Long Side Fit (BLSF)': blsf,
}

rows = []
for name, data in methods.items():
    rows.append({
        'Method': name,
        'Mean (%)':   data.mean() * 100,
        'Std (%)':    data.std(ddof=1) * 100,
        'Min (%)':    data.min() * 100,
        'Max (%)':    data.max() * 100,
        'Median (%)': np.median(data) * 100,
    })

desc = pd.DataFrame(rows).set_index('Method')
print('=== Descriptive Statistics (n=16 flights) ===')
print(desc.to_string(float_format=lambda x: f'{x:.2f}'))

In [ ]:
# ================================================================
# WILCOXON SIGNED-RANK TESTS  (one-sided: heuristic > historical)
# H0: median difference = 0
# H1: heuristic utilisation > historical utilisation
# ================================================================

print('=== Wilcoxon Signed-Rank Tests (heuristic > Historical baseline) ===')
print(f'n = {len(historical)}, significance level alpha = 0.05\n')

wilcoxon_results = []
for name, data in [('BL', bl), ('BAF', baf), ('BLSF', blsf)]:
    diff = data - historical
    stat, p = stats.wilcoxon(diff, alternative='greater')
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    wilcoxon_results.append({
        'Comparison': f'{name} vs. Historical',
        'W-statistic': stat,
        'p-value': p,
        'Significance': sig,
        'Mean diff (pp)': diff.mean() * 100
    })
    print(f'{name} vs. Historical: W={stat:.1f}, p={p:.4f} {sig}, mean diff={diff.mean()*100:+.2f} pp')

wilcoxon_df = pd.DataFrame(wilcoxon_results).set_index('Comparison')
print('\nLegend: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')

In [ ]:
# ================================================================
# FIGURE: Per-flight utilisation comparison (for the paper)
# ================================================================

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(16)
w = 0.20

bars = [
    (historical, 'Historical', '#888888', -1.5*w),
    (bl,         'BL',         '#1f77b4',  -0.5*w),
    (baf,        'BAF',        '#ff7f0e',   0.5*w),
    (blsf,       'BLSF',       '#2ca02c',   1.5*w),
]

for data, label, color, offset in bars:
    ax.bar(x + offset, data * 100, w, label=label, color=color, alpha=0.85)

ax.set_xlabel('Flight', fontsize=12)
ax.set_ylabel('Volumetric Utilisation (%)', fontsize=12)
ax.set_title('Per-flight Volumetric Utilisation by Packing Method', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(flight_ids, rotation=45, ha='right')
ax.set_ylim(50, 100)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='lower right', framealpha=0.9)
ax.axhline(historical.mean() * 100, color='#888888', linestyle='--',
           linewidth=1.2, label='Historical mean')
ax.axhline(bl.mean() * 100,         color='#1f77b4', linestyle='--',
           linewidth=1.2, label='BL mean')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Paper/fig_per_flight_utilization.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../Paper/fig_per_flight_utilization.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved to Paper/')

In [ ]:
# ================================================================
# LATEX TABLE — publication-ready output for Table 6
# Copy-paste into the manuscript
# ================================================================

def fmt(val, bold=False):
    s = f'{val:.2f}'
    return f'\\textbf{{{s}}}' if bold else s

best_mean = max(bl.mean(), baf.mean(), blsf.mean()) * 100

latex = r"""
\begin{table}[ht]
\caption{Comparative volumetric utilisation across 16 charter flights.
         Values in \%. Statistical significance of the improvement over
         the historical baseline is assessed via a one-sided Wilcoxon
         signed-rank test ($n=16$, $\alpha=0.05$).}
\label{tab:comparative}
\centering
\begin{tabular}{lrrrrrrc}
\toprule
Method & Mean & Std & Min & Max & Median
       & $\Delta$\,mean & $p$-value \\
       &      &     &     &     &
       & (pp) & \\
\midrule
"""

for name, data, wrow in [
    ('Historical (baseline)', historical, None),
    ('Bottom-Left (BL)',      bl,   wilcoxon_results[0]),
    ('Best Area Fit (BAF)',   baf,  wilcoxon_results[1]),
    ('Best Long Side Fit (BLSF)', blsf, wilcoxon_results[2]),
]:
    is_best = abs(data.mean()*100 - best_mean) < 0.001
    mean_s = fmt(data.mean()*100, bold=is_best)
    std_s  = f'{data.std(ddof=1)*100:.2f}'
    min_s  = f'{data.min()*100:.2f}'
    max_s  = f'{data.max()*100:.2f}'
    med_s  = f'{np.median(data)*100:.2f}'
    if wrow is None:
        delta_s = '—'
        p_s     = '—'
    else:
        delta_s = f"{wrow['Mean diff (pp)']:+.2f}"
        p_val   = wrow['p-value']
        sig     = wrow['Significance']
        p_s     = f'{p_val:.4f}\\,{sig}' if p_val >= 0.0001 else f'$<$0.0001\\,{sig}'
    latex += f'{name} & {mean_s} & {std_s} & {min_s} & {max_s} & {med_s} & {delta_s} & {p_s} \\\\\n'

latex += r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
  \small
  \item pp = percentage points relative to historical mean.
  \item Significance codes: *** $p<0.001$, ** $p<0.01$, * $p<0.05$, ns = not significant.
\end{tablenotes}
\end{table}
"""

print(latex)

In [ ]:
# ================================================================
# PER-FLIGHT DELTA TABLE  (for Table 10 in appendix)
# ================================================================

df_flights = pd.DataFrame({
    'Flight': flight_ids,
    'Historical (%)': historical * 100,
    'BL (%)':         bl * 100,
    'BAF (%)':        baf * 100,
    'BLSF (%)':       blsf * 100,
    'Δ BL (pp)':      (bl - historical) * 100,
    'Δ BAF (pp)':     (baf - historical) * 100,
    'Δ BLSF (pp)':    (blsf - historical) * 100,
}).set_index('Flight')

print('=== Per-flight results ===')
print(df_flights.to_string(float_format=lambda x: f'{x:.2f}'))

# Save to CSV for easy import into Word/Excel
df_flights.to_csv('../Paper/per_flight_results.csv', float_format='%.4f')
print('\nSaved to Paper/per_flight_results.csv')